# 04 – CBS drivers (Kerncijfers wijken en buurten 2019–2026)

**Goal:** build a table of neighbourhood characteristics (drivers) per buurt per year, on 2023 buurt codes.
**Input:** `kwb-2019.xlsx` … `kwb2026.xlsx`, `Toelichting-variabelen-kwb-2026.pdf`
**Outputs:** `Data/processed/cbs_kwb_buurt_2019_2026.csv` (own codes per year), `Data/processed/cbs_buurt_panel_2023codes.csv` (final panel)

## Cleaning
- Only rows with `recs = Buurt`. CBS `.` = unknown/secret → missing. Decimal commas and dots both handled (text work first, then missing, otherwise `.str` fails on all-empty columns).
- `g_woz` (2019) and `g_wozbag` (2020+) combined into `woz` (× €1,000).

## Availability
- 2026: almost empty (not yet published). Income ~2 years delayed. `g_ele`/`g_gas` until 2024.
- Building age: `p_bjj2k/p_bjo2k` until 2023, `p_bj_me10/p_bj_mi10` from 2024 (different definitions; kept separate).
- Dropped: `a_lp_pub` (0% filled at buurt level), `p_stadsv` (5–8% filled).

## Buurt codes
- Codes change over the years: 2023 codes found in 77% (2019) → 90% (2022) → 98% (2024–2026).
- **Decision:** direct code match; gaps filled within each buurt with the latest known value (forward), then the first known value (backward). Flag `cbs_code_found` shows which rows are real.
- Assumption: CBS drivers change slowly. Limitation: filled early years contain some "later" information.
- Remaining gaps (CBS privacy suppression in small buurten) are left missing → LightGBM handles missing values.

## Energy and solar variables (2024 only, used as fixed per buurt)
- `p_won_zs` = % homes with solar panels; `p_won_ev` = % homes mainly electrically heated (e.g. heat pumps); `p_won_z` = % gas-free homes; `p_won_m` = % gas-heated homes. Suppressed below 100 homes → 72% filled.
- `g_ele_tr` = average feed-in per home: median 1,020 kWh; feed-in / delivery = 0.36 → solar has made the local grid two-way.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

RAW = Path("../Data/raw")
PROCESSED = Path("../Data/processed")
KWB = RAW / "2 CBS" / "1 kwb Raw"

files = sorted(KWB.glob("kwb*.xlsx"))

# candidate drivers we would like to use
candidates = [
    "a_inw", "a_hh", "g_hhgro", "bev_dich", "a_woning", "a_bedv", "a_opp_ha", "a_lan_ha",
    "ste_mvs", "ste_oad",
    "g_woz", "g_wozbag", "p_1gezw", "p_mgezw", "p_koopw", "p_huurw", "p_leegsw",
    "p_bjj2k", "p_bjo2k", "p_bj_me10", "p_bj_mi10",
    "g_ele", "g_gas", "p_stadsv",
    "g_ink_pi", "a_pau", "g_pau_hh",
    "a_lp_pub", "p_won_z", "p_won_m", "p_won_zs", "p_won_ev",
]

availability = {}
for f in files:
    year = int(re.search(r"(20\d{2})", f.name).group(1))
    cols = pd.read_excel(f, nrows=0).columns.str.strip()
    availability[year] = ["✓" if c in cols else "-" for c in candidates]
    print("checked", f.name)

matrix = pd.DataFrame(availability, index=candidates)
print()
print(matrix.to_string())

checked kwb-2019.xlsx
checked kwb-2020.xlsx
checked kwb-2021.xlsx
checked kwb-2022.xlsx
checked kwb2023.xlsx
checked kwb2024.xlsx
checked kwb2025.xlsx
checked kwb2026.xlsx

          2019 2020 2021 2022 2023 2024 2025 2026
a_inw        ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
a_hh         ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
g_hhgro      ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
bev_dich     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
a_woning     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
a_bedv       ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
a_opp_ha     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
a_lan_ha     ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
ste_mvs      ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
ste_oad      ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
g_woz        ✓    -    -    -    -    -    -    -
g_wozbag     -    ✓    ✓    ✓    ✓    ✓    ✓    ✓
p_1gezw      ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
p_mgezw      ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
p_koopw      ✓    ✓    ✓    ✓    ✓    ✓    ✓    ✓
p_huurw      ✓    ✓    ✓   

In [2]:
def clean_num(s):
    s = s.astype(str).str.strip().str.replace(",", ".", regex=False)   # text work first
    s = s.replace({".": np.nan, "": np.nan, "nan": np.nan})             # then mark missing
    return pd.to_numeric(s, errors="coerce")

parts = []
for f in files:
    year = int(re.search(r"(20\d{2})", f.name).group(1))
    df = pd.read_excel(f, dtype=str)
    df.columns = df.columns.str.strip()
    df = df[df["recs"].str.strip() == "Buurt"]                    # buurt rows only

    out = pd.DataFrame({
        "buurtcode": df["gwb_code"].str.strip(),
        "buurtnaam": df["regio"].str.strip(),
        "gemeentenaam": df["gm_naam"].str.strip(),
        "year": year,
    })
    for c in candidates:
        if c in df.columns:
            out[c] = clean_num(df[c])
    parts.append(out)
    print(f.name, "->", len(out), "buurten")

kwb = pd.concat(parts, ignore_index=True)
kwb["woz"] = kwb["g_woz"].fillna(kwb["g_wozbag"])                 # one home-value column
kwb = kwb.drop(columns=["g_woz", "g_wozbag"])

# % of buurten with a value, per column per year
num_cols = [c for c in kwb.columns if c not in ["buurtcode", "buurtnaam", "gemeentenaam", "year"]]
fill = kwb.groupby("year")[num_cols].apply(lambda d: d.notna().mean() * 100).round(0).T
print()
print(fill.to_string())

kwb.to_csv(PROCESSED / "cbs_kwb_buurt_2019_2026.csv", index=False)
print("\nSaved: cbs_kwb_buurt_2019_2026.csv", kwb.shape)

kwb-2019.xlsx -> 13594 buurten
kwb-2020.xlsx -> 13808 buurten
kwb-2021.xlsx -> 14080 buurten
kwb-2022.xlsx -> 14317 buurten
kwb2023.xlsx -> 14421 buurten


C:\Users\Karam\AppData\Local\Temp\ipykernel_23524\2102840781.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = s.replace({".": np.nan, "": np.nan, "nan": np.nan})             # then mark missing


kwb2024.xlsx -> 14574 buurten


C:\Users\Karam\AppData\Local\Temp\ipykernel_23524\2102840781.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = s.replace({".": np.nan, "": np.nan, "nan": np.nan})             # then mark missing
C:\Users\Karam\AppData\Local\Temp\ipykernel_23524\2102840781.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = s.replace({".": np.nan, "": np.nan, "nan": np.nan})             # then mark missing
C:\Users\Karam\AppData\Local\Temp\ipykernel_23524\2102840781.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and 

kwb2025.xlsx -> 14729 buurten


C:\Users\Karam\AppData\Local\Temp\ipykernel_23524\2102840781.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = s.replace({".": np.nan, "": np.nan, "nan": np.nan})             # then mark missing
C:\Users\Karam\AppData\Local\Temp\ipykernel_23524\2102840781.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = s.replace({".": np.nan, "": np.nan, "nan": np.nan})             # then mark missing
C:\Users\Karam\AppData\Local\Temp\ipykernel_23524\2102840781.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and 

kwb2026.xlsx -> 14820 buurten

year        2019   2020   2021   2022   2023   2024   2025   2026
a_inw      100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0
a_hh       100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0
g_hhgro     97.0   97.0   97.0   97.0   97.0   97.0   97.0   97.0
bev_dich    95.0   95.0   95.0   95.0   95.0   95.0   95.0   95.0
a_woning   100.0  100.0  100.0  100.0  100.0  100.0  100.0    0.0
a_bedv     100.0  100.0  100.0  100.0  100.0  100.0  100.0    0.0
a_opp_ha   100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0
a_lan_ha   100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0
ste_mvs    100.0  100.0   99.0   99.0   99.0   99.0   99.0   99.0
ste_oad    100.0  100.0   99.0   99.0   99.0   99.0   99.0   99.0
p_1gezw     89.0   89.0   89.0   89.0   89.0   89.0   89.0    0.0
p_mgezw     89.0   89.0   89.0   89.0   89.0   89.0   89.0    0.0
p_koopw     89.0   89.0   89.0   89.0   89.0   89.0   89.0    0.0
p_huurw     89.0   89.0   89.0   89.0   89.0 

In [3]:
# the buurten we actually need: those in the Liander consumption data (2023 codes)
elec_bu = pd.read_csv(PROCESSED / "buurt_year_electricity.csv")
liander_buurten = set(elec_bu["buurtcode"].unique())
print("Liander buurten (2023 codes):", len(liander_buurten))

codes_2023 = set(kwb.loc[kwb["year"] == 2023, "buurtcode"])
print("Liander buurten found in CBS 2023:", len(liander_buurten & codes_2023))

print("\nShare of Liander buurten whose 2023 code exists in each CBS year:")
for year in sorted(kwb["year"].unique()):
    codes_year = set(kwb.loc[kwb["year"] == year, "buurtcode"])
    found = len(liander_buurten & codes_year)
    print(f"  {year}: {found:,} of {len(liander_buurten):,} ({100 * found / len(liander_buurten):.1f}%)")

Liander buurten (2023 codes): 5672
Liander buurten found in CBS 2023: 5672

Share of Liander buurten whose 2023 code exists in each CBS year:
  2019: 4,346 of 5,672 (76.6%)
  2020: 4,620 of 5,672 (81.5%)
  2021: 4,768 of 5,672 (84.1%)
  2022: 5,109 of 5,672 (90.1%)
  2023: 5,672 of 5,672 (100.0%)
  2024: 5,589 of 5,672 (98.5%)
  2025: 5,565 of 5,672 (98.1%)
  2026: 5,558 of 5,672 (98.0%)


In [4]:
# features to keep (drop the two unusable ones)
feat = [c for c in num_cols if c not in ["a_lp_pub", "p_stadsv"]]

# 1. CBS rows of the Liander buurten only
k = kwb[kwb["buurtcode"].isin(liander_buurten)][["buurtcode", "year"] + feat]
k = k.drop_duplicates(["buurtcode", "year"])

# 2. full grid: every Liander buurt x every year
years = sorted(kwb["year"].unique())
grid = pd.MultiIndex.from_product([sorted(liander_buurten), years],
                                  names=["buurtcode", "year"]).to_frame(index=False)
panel = grid.merge(k, on=["buurtcode", "year"], how="left")

# 3. flag: did this buurt-year have real CBS data?
panel["cbs_code_found"] = panel["a_inw"].notna()

before = panel[feat].notna().mean() * 100

# 4. fill gaps within each buurt: forward (latest known), then backward (first known)
panel = panel.sort_values(["buurtcode", "year"])
panel[feat] = panel.groupby("buurtcode")[feat].ffill()
panel[feat] = panel.groupby("buurtcode")[feat].bfill()

after = panel[feat].notna().mean() * 100

print("Panel shape:", panel.shape)
print("\n% filled per feature (before -> after):")
print(pd.DataFrame({"before": before, "after": after}).round(1).to_string())

print("\nReal CBS data per year (%):")
print((panel.groupby("year")["cbs_code_found"].mean() * 100).round(1).to_string())

panel.to_csv(PROCESSED / "cbs_buurt_panel_2023codes.csv", index=False)
print("\nSaved: cbs_buurt_panel_2023codes.csv")

Panel shape: (45376, 32)

% filled per feature (before -> after):
           before  after
a_inw        90.9  100.0
a_hh         90.9  100.0
g_hhgro      89.4   98.6
bev_dich     87.8   96.8
a_woning     78.6  100.0
a_bedv       78.6  100.0
a_opp_ha     90.9  100.0
a_lan_ha     90.9  100.0
ste_mvs      90.8   99.9
ste_oad      90.8   99.9
p_1gezw      71.6   91.1
p_mgezw      71.6   91.1
p_koopw      71.6   91.1
p_huurw      71.6   91.1
p_leegsw     71.6   91.1
p_bjj2k      49.3   90.8
p_bjo2k      49.3   90.8
g_ele        63.2   95.6
g_gas        62.8   95.4
g_ink_pi     47.1   85.1
a_pau        78.6  100.0
g_pau_hh     63.9   83.2
p_bj_me10    22.3   89.4
p_bj_mi10    22.3   89.4
p_won_z       9.0   72.0
p_won_m       9.0   72.0
p_won_zs      9.0   72.0
p_won_ev      9.0   72.0
woz          70.7   90.4

Real CBS data per year (%):
year
2019     76.6
2020     81.5
2021     84.1
2022     90.1
2023    100.0
2024     98.5
2025     98.1
2026     98.0

Saved: cbs_buurt_panel_2023codes.csv


In [5]:
k24 = pd.read_excel(KWB / "kwb2024.xlsx", dtype=str)
k24.columns = k24.columns.str.strip()
k24 = k24[k24["recs"].str.strip() == "Buurt"]

extra = pd.DataFrame({
    "buurtcode": k24["gwb_code"].str.strip(),
    "g_ele_tr": clean_num(k24["g_ele_tr"]),
})

panel = panel.drop(columns=["g_ele_tr"], errors="ignore").merge(extra, on="buurtcode", how="left")

# check
print(f"g_ele_tr filled: {100 * panel['g_ele_tr'].notna().mean():.1f}%")
print("\nFeed-in per home (kWh), 2024:")
print(panel.loc[panel["year"] == 2024, "g_ele_tr"].describe().round(0).to_string())

# how does feed-in compare with delivery to homes?
p24 = panel[panel["year"] == 2024]
ratio = (p24["g_ele_tr"] / p24["g_ele"]).median()
print(f"\nMedian feed-in / delivery ratio: {ratio:.2f}")

panel.to_csv(PROCESSED / "cbs_buurt_panel_2023codes.csv", index=False)
print("\nSaved again with g_ele_tr:", panel.shape)

g_ele_tr filled: 93.5%

Feed-in per home (kWh), 2024:
count    5305.0
mean     1142.0
std       825.0
min         0.0
25%       530.0
50%      1020.0
75%      1610.0
max      6930.0

Median feed-in / delivery ratio: 0.36

Saved again with g_ele_tr: (45376, 33)


In [6]:
stress = pd.read_csv(PROCESSED / "buurt_transformer_stress.csv")
p24 = panel[panel["year"] == 2024].merge(stress, on="buurtcode", how="inner")
print("Buurten with CBS 2024 data and transformer data:", len(p24))

# 1. median of each feature: stressed vs not stressed
feats = ["p_won_zs", "g_ele_tr", "p_won_ev", "p_1gezw", "woz", "bev_dich", "g_ele", "a_woning", "n_transformers"]
comp = p24.groupby("stress_nbl")[feats].median().T
comp.columns = ["not_stressed", "stressed"]
comp["ratio"] = comp["stressed"] / comp["not_stressed"]
print("\nMedian per group (ratio > 1 = higher in stressed buurten):")
print(comp.round(2).to_string())

# 2. stress rate and average utilisation per solar group (5 equal groups)
for col in ["p_won_zs", "g_ele_tr"]:
    p24["group"] = pd.qcut(p24[col], 5, labels=["Q1 low", "Q2", "Q3", "Q4", "Q5 high"])
    g = p24.groupby("group", observed=True).agg(
        buurten=("buurtcode", "size"),
        stress_rate_pct=("stress_nbl", "mean"),
        avg_util=("util_weighted", "mean"),
    )
    g["stress_rate_pct"] *= 100
    print(f"\nBy {col} (quintiles):")
    print(g.round(2).to_string())

Buurten with CBS 2024 data and transformer data: 5451

Median per group (ratio > 1 = higher in stressed buurten):
                not_stressed  stressed  ratio
p_won_zs                43.0      35.0   0.81
g_ele_tr              1040.0     755.0   0.73
p_won_ev                 7.0       6.0   0.86
p_1gezw                 86.0      73.5   0.85
woz                    418.0     433.0   1.04
bev_dich              2268.0    3281.0   1.45
g_ele                 2830.0    2630.0   0.93
a_woning               336.5     688.0   2.04
n_transformers           4.0       7.0   1.75

By p_won_zs (quintiles):
         buurten  stress_rate_pct  avg_util
group                                      
Q1 low       807            19.83      0.49
Q2           834             9.83      0.48
Q3           818            10.64      0.46
Q4           772             8.94      0.43
Q5 high      754             6.63      0.42

By g_ele_tr (quintiles):
         buurten  stress_rate_pct  avg_util
group                 

In [7]:
elec = pd.read_csv(PROCESSED / "buurt_year_electricity.csv")
e24 = elec[elec["year"] == 2024][["buurtcode", "connections", "total_kwh"]]
q = p24.merge(e24, on="buurtcode", how="left")

# load RELATIVE to the grid infrastructure
q["conn_per_trafo"]        = q["connections"] / q["n_transformers"]
q["kwh_per_kw_capacity"]   = q["total_kwh"] / q["capacity_kw_total"]
q["solar_homes_per_trafo"] = (q["p_won_zs"] / 100) * q["a_woning"] / q["n_transformers"]
q["feedin_kwh_per_kw_cap"] = q["g_ele_tr"] * q["a_woning"] / q["capacity_kw_total"]

rel = ["conn_per_trafo", "kwh_per_kw_capacity", "solar_homes_per_trafo", "feedin_kwh_per_kw_cap"]

comp2 = q.groupby("stress_nbl")[rel].median().T
comp2.columns = ["not_stressed", "stressed"]
comp2["ratio"] = comp2["stressed"] / comp2["not_stressed"]
print("Median per group:")
print(comp2.round(2).to_string())

for col in rel:
    q["group"] = pd.qcut(q[col], 5, labels=["Q1 low", "Q2", "Q3", "Q4", "Q5 high"])
    g = q.groupby("group", observed=True).agg(
        buurten=("buurtcode", "size"),
        stress_rate_pct=("stress_nbl", "mean"),
        avg_util=("util_weighted", "mean"),
    )
    g["stress_rate_pct"] *= 100
    print(f"\nBy {col}:")
    print(g.round(2).to_string())

Median per group:
                       not_stressed  stressed  ratio
conn_per_trafo               110.14    128.52   1.17
kwh_per_kw_capacity         1261.77   1398.79   1.11
solar_homes_per_trafo         47.20     31.47   0.67
feedin_kwh_per_kw_cap        230.00    147.16   0.64

By conn_per_trafo:
         buurten  stress_rate_pct  avg_util
group                                      
Q1 low      1087             7.73      0.30
Q2          1086             9.67      0.35
Q3          1087            10.67      0.43
Q4          1085            11.89      0.49
Q5 high     1086            11.23      0.54

By kwh_per_kw_capacity:
         buurten  stress_rate_pct  avg_util
group                                      
Q1 low      1087             7.54      0.29
Q2          1086             8.38      0.36
Q3          1086            11.14      0.43
Q4          1086            11.23      0.49
Q5 high     1086            12.89      0.54

By solar_homes_per_trafo:
         buurten  stress_rate

## First exploration: what separates stressed buurten? (2024, 5,451 buurten)
- **Per home, solar is NOT higher in stressed buurten:** homes with solar 35% vs 43%, feed-in 755 vs 1,040 kWh. Stress rate is highest in the lowest solar quintile (19.8%).
- Stressed buurten are **larger and denser**: 2× the homes, 1.45× the density, fewer single-family homes.
- **Load relative to infrastructure gives a clear, stepwise signal:**
  - connections per transformer: utilisation 0.30 → 0.54 (Q1 → Q5)
  - kWh per kW capacity: utilisation 0.29 → 0.54, stress rate 7.5% → 12.9%
- Solar feed-in per kW capacity: utilisation rises (0.36 → 0.53), stress rate does not.
- Possible explanations: per-home % is a proxy for housing type (apartments rarely have own panels); grid may already be reinforced in solar-heavy areas (reverse causality); `stress_nbl` favours buurten with many transformers.
- **Decisions:** use "relative to infrastructure" features; keep solar features and let the model/SHAP judge; build Model B both as classification (`stress_nbl`) and regression (`util_weighted`, size-independent).

In [8]:
cars = pd.read_excel(RAW / "2 CBS" / "3 Passenger car fuel.xlsx", header=None)
print("Shape:", cars.shape)
pd.set_option("display.max_columns", 20)
print(cars.to_string())

Shape: (16, 11)
                                                                  0       1   2          3          4          5          6          7          8          9          10
0                                                           Tabel 1      NaN NaN        NaN        NaN        NaN        NaN        NaN        NaN        NaN        NaN
1   Personenauto's naar brandstofsoort/aandrijflijn, stand 1 januari     NaN NaN        NaN        NaN        NaN        NaN        NaN        NaN        NaN        NaN
2                                                                NaN     NaN NaN     2019.0     2020.0     2021.0     2022.0     2023.0     2024.0     2025.0     2026.0
3                                                                NaN  aantal NaN        NaN        NaN        NaN        NaN        NaN        NaN        NaN        NaN
4                                                                NaN     NaN NaN        NaN        NaN        NaN        NaN        NaN    

## Transport electrification (car fuel, CBS/RDW)
- `3 Passenger car fuel.xlsx` is **national only** (one row per fuel type, 1 January 2019–2026) → cannot explain differences between buurten; not used as a model feature.
- Trend: fully electric cars 44,581 (0.5%) → 694,602 (7.4%) of all cars; plug-in hybrids 93,839 → 523,773; together ~1 in 8 cars can charge from the grid in 2026.
- Used as national context (dashboard and report). Limitation: no local EV data at buurt level (`a_lp_pub` empty, `a_bst_nb` mixes diesel and electric). Future work: municipality-level EV shares from CBS StatLine.